In [205]:
import confnotebook

In [206]:
from pathlib import Path

source = Path("../examples/test/full/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 10
[1] 126164
[2] 14964427_Енисейская ТГК-13-БРАЗ
[3] 14976087_АвеларСолар Тех-БРАЗ-1
[4] 15120979_Форвард Энерго-БРАЗ-1
[5] 15235008_ОГК-2-БРАЗ-1
[6] 25
[7] 33
[8] 4
[9] 44
[10] 7-1
[11] АС ВНИИМ Менделеева Д.И. - БРАЗ на 31.12.24
[12] АС ВНИИМ Менделеева Д.И. - РУ на 31.12.24
[13] АС КРЕЗОЛ-САЗ на 31.08.25
[14] АС Охрана Металлург-САЗ на 31.12.25
[15] АС РУ- ВОСЬМОЙ ВЕТРОПАРК
[16] АС Фрейт Линк-БРАЗ на 30.09.25
[17] АСР СДД 2 кв.2024 (подп. к-а)
[18] Акт сверки взаимных расчетов №00000379931 от 30.04.2024
[19] Акт сверки №0000
[20] Акт сверки №MOW00-0087974   от 10.06.2024
[21] Акт сверки №ТРБП-000006 от 10.01.2024
[22] Браз-Юнигрин Пауэр
[23] ЕВР-НКАЗ
[24] Неформализованный_первичный_документ_23_ИИА_03_00342_от_31_03
[25] Неформализованный_первичный_документ_23_ИИА_03_02596_от_31_03
[26] Неформализованный_первичный_документ_23_ИИА_03_03623_от_31_03
[27] Неформализованный_первичный_документ_23_ИИА_06_01794_от_30_06
[28] ПР_АС КРЕЗОЛ-САЗ на 31.08.25
[29] ПР_АС Фрейт Линк-

In [207]:
IDX_FILE = 22


file = files[IDX_FILE]

In [208]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline()

document = pipeline.build(file.read_bytes())

Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-04-25 17:12:21.551 | INFO     | vision_core.pipelines.build_document:build:89 - Обработка страницы 0 с dpi 200...
2026-04-25 17:12:21.580 | INFO     | vision_core.pipelines.build_document:_process_page:146 - Предобработка изображения...
2026-04-25 17:12:21.585 | INFO     | vision_core.pipelines.build_document:_process_page:148 - Предобработка завершена.
2026-04-25 17:12:21.585 | INFO     | vision_core.pipelines.build_document:_process_page:150 - Коррекция ориентации и наклона...
2026-04-25 17:12:21.592 | DEBUG    | vision_core.preprocessor.image_orientation:process:41 - Ориентация страницы: 0° с точностью 0.8993
2026-04-25 17:12:21.601 | DEBUG    | vision_core.preprocessor.image_orientation:_correct_perspective_from_table

In [209]:

# Собираем все сырые OCR-тексты из document
raw_texts: list[str] = []

for page in document.pages:
    for para in page.paragraphs:
        if para.text:
            raw_texts.append(para.text)
    for table in page.tables:
        for cell in table.cells:
            if cell.value:
                raw_texts.append(cell.value)

print(f"Всего строк: {len(raw_texts)}")
print("Примеры:")
for t in raw_texts:
    print(repr(t))


Всего строк: 42
Примеры:
'Приложение 2 к Договору ӧ предоставлении мӧщности квалифицированных генерирующих объектов; функционирующих на основе использования возобновляемых источников энергии'
'АКТ СВЕРКИ РАСЧЕТОВ'
'по Договору о предоставлении мощности квалифицированных генерирующих.объектов, функционирующих на основе использования возобновляемых источников энергии N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ӧт 20:10.2021 г. за 1 квартал 2025 г.'
'00 "Юнигрин Паузр" Мы, нижеподгисавшиеся, , с одной стороны, и Публичное акционерное общество "РуСАЛ Братский алюминиевый завод" , с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее:'
'01.04.2025r.'
'(py6.)'
'000."Юнигрин Паузр"'
'От Продааца'
'От Локупателя Публичное акционерное общество "РуСАл Братский алюминиевый завод"'
'По дов-ти №124.1/2024.от 01,11:2024 Савина Т.Г:'
'Ведущий бухгалтер по учету операций на ОРЭМ Савина Т.Г.:'
'_.'
'По данным Продавца'
'(py6.)\nПо данным Покулател

In [210]:
import unicodedata
from collections import Counter

# Считаем все символы по всему корпусу
char_counter = Counter(ch for text in raw_texts for ch in text)

# Выводим отсортированные по частоте, с unicode-именем
print(f"{'Char':>6}  {'Code':>7}  {'Count':>6}  Name")
print("-" * 60)
for ch, count in char_counter.most_common():
    code = f"U+{ord(ch):04X}"
    name = unicodedata.name(ch, "?")
    display = repr(ch) if ch in ("\n", "\t", " ") else ch
    print(f"{display:>6}  {code:>7}  {count:>6}  {name}")


  Char     Code   Count  Name
------------------------------------------------------------
   ' '   U+0020     141  SPACE
     о   U+043E     124  CYRILLIC SMALL LETTER O
     н   U+043D      80  CYRILLIC SMALL LETTER EN
     а   U+0430      75  CYRILLIC SMALL LETTER A
     и   U+0438      72  CYRILLIC SMALL LETTER I
     е   U+0435      67  CYRILLIC SMALL LETTER IE
     0   U+0030      67  DIGIT ZERO
     т   U+0442      57  CYRILLIC SMALL LETTER TE
     в   U+0432      54  CYRILLIC SMALL LETTER VE
     р   U+0440      44  CYRILLIC SMALL LETTER ER
     с   U+0441      43  CYRILLIC SMALL LETTER ES
     л   U+043B      34  CYRILLIC SMALL LETTER EL
     .   U+002E      32  FULL STOP
     у   U+0443      29  CYRILLIC SMALL LETTER U
     д   U+0434      27  CYRILLIC SMALL LETTER DE
     2   U+0032      26  DIGIT TWO
     п   U+043F      25  CYRILLIC SMALL LETTER PE
     к   U+043A      24  CYRILLIC SMALL LETTER KA
     ,   U+002C      24  COMMA
     1   U+0031      19  DIGIT ONE
     м   U

In [211]:
import unicodedata


def _build_extended_cyrillic_map() -> dict:
    russian = set(range(0x0410, 0x0450)) | {0x0401, 0x0451}  # А-Яа-я + Ёё
    result = {}
    for cp in range(0x0400, 0x0500):
        if cp in russian:
            continue
        decomposed = unicodedata.normalize("NFD", chr(cp))
        base = decomposed[0]
        if ord(base) in russian:
            result[cp] = base  # Ӧ -> О, Ӓ -> А, Ӗ -> Е ...
    return result

NORM_MAP = str.maketrans(
    {
        # Двойные кавычки -> стандартная “
        0x00AB: '"',  # «
        0x00BB: '"',  # »
        0x201C: '"',  # “
        0x201D: '"',  # ”
        0x201E: '"',  # „
        0x003C: '"',  # <
        0x003E: '"',  # >
        # Одиночные кавычки и OCR-мусор
        0x0027: ' ',  # ‘
        0x0060: ' ',  # `
        0x00B7: ' ',  # ·
        0x2018: ' ',  # ‘
        0x2019: ' ',  # ‘
        0x201A: ' ',  # ‚
        0x2039: ' ',  # ‹
        0x203A: ' ',  # ›
        0x2021: ' ',  # ‡
        0x2020: ' ',  # †

        0x0404: "Е",  # Є  Ukrainian
        0x0454: "Е",  # є
        0x0490: "Г",  # Ґ  Ukrainian
        0x0491: "Г",  # ґ
        0x040E: "У",  # Ў  Belarusian
        0x045E: "У",  # ў

        0x040F: "Т",  # Џ  -> Т (Тел.)
        0x040C: "К",  # Ќ
        0x0403: "Г",  # Ѓ
        0x0405: "З",  # Ѕ
        0x0408: "Й",  # Ј
        0x0498: "З",  # ҙ

        0x04AE: "У",  # Ү
        0x04AF: "У",  # ү
        0x04B0: "У",  # Ұ
        0x04B1: "У",  # ұ
        0x0492: "Г",  # Ғ
        0x0493: "Г",  # ғ
        0x049A: "К",  # Қ
        0x049B: "К",  # қ
        0x04A2: "Н",  # Ң
        0x04A3: "Н",  # ң
        0x04E8: "О",  # Ө
        0x04E7: "О",  # ө
        0x04E9: "О",  # ө
        0x04E4: "И",  # Ӥ
        0x04E5: "И",  # ӥ
        0x04E2: "И",  # Ӣ
        0x04E3: "И",  # ӣ
        0x0499: "З",  # ҙ

        0x0462: "Е",  # Ѣ  ять
        0x0472: "Ф",  # Ѳ  фита
        0x0474: "И",  # Ѵ  ижица

        0x005B: None,  # [
        0x005D: None,  # ]
        0x007C: None,  # |

        ord("Ё"): "Е",
    }

)

def normalize_text(text: str) -> str:
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.translate(m)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t.upper())))

Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ.ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20:10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, , С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД" , С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025R.'
'(PY6.)'
'000."ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1/2024.ОТ 01,11:2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_.'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.)\nПО ДАННЫМ ПОКУЛАТЕЛЯ

In [212]:
import re

_RULES_SPACE: list[tuple[re.Pattern, str, str]] = [
    (re.compile(r'\s+', re.UNICODE),   ' ',    'collapse_spaces'), # "слово   слово" -> "слово слово"
    (re.compile(r'\s+([,.:;])'),       r'\1',  'space_before_punct'), # "слово , слово" -> "слово, слово"
    (re.compile(r'([,;:])(?=[^\s\d])'), r'\1 ', 'space_after_punct'), # "слово,слово" -> "слово, слово"
    (re.compile(r'(?<=\w\w)\.(?=[А-ЯЁA-Za-z])'), '. ', 'space_after_dot'), # 0.00 -> 0.00, но "слово.слово" -> "слово. слово"
]

_RULES_QUOTES: list[tuple[re.Pattern, str, str]] = [
    (re.compile(r'"{2,}'),         '"',      'collapse_quotes'), # """" -> "
    (re.compile(r'(\S)"(?=\w)'),   r'\1 "',  'space_before_quote'), # слово" -> слово "
    (re.compile(r'\.{2,}'),        '.',      'collapse_dots'), # ....... -> .
    (re.compile(r',{2,}'),         ',',      'collapse_commas'), # ,,,,, -> ,
    (re.compile(r'(?<![А-ЯЁA-Z0-9.])[.:]'), '', 'leading_punct'), # .0.00 -> 0.00 :0.00 -> 0.00
    (re.compile(r'([,;:.!])\s*\1+'), r'\1', 'collapse_dup_punct'),

]

def _apply_rules(text: str, rules: list[tuple[re.Pattern, str, str]]) -> str:
    for pattern, repl, _ in rules:
        text = pattern.sub(repl, text)
    return text


def normalize_text(text: str) -> str:
    
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.upper().translate(m)

    
    text = _apply_rules(text, _RULES_SPACE)
    text = _apply_rules(text, _RULES_QUOTES)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t)))

Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ. ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20:10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025R.'
'(PY6.)'
'000. "ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1/2024. ОТ 01,11:2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.) ПО ДАННЫМ ПОКУЛАТЕЛЯ'


In [213]:
# Символы, которые OCR путает с цифрами (после upper())
_LAT_TO_DIGIT = str.maketrans("OI", "01")

# Числовой контекст: последовательность цифро-подобных символов через . или ,
# \b не используем — он не работает с кириллицей, поэтому (?<!\w)
RE_NUMERIC = re.compile(r'(?<![А-ЯЁA-Z])(?:[0-9OI]+[.,])+[0-9OI]+(?![А-ЯЁA-Z])')

def _fix_numeric(m: re.Match) -> str:
    return m.group(0).translate(_LAT_TO_DIGIT)

def normalize_text(text: str) -> str:
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.upper().translate(m)
    
    text = RE_NUMERIC.sub(_fix_numeric, text)
    text = _apply_rules(text, _RULES_SPACE)
    text = _apply_rules(text, _RULES_QUOTES)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t)))

Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ. ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20:10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025R.'
'(PY6.)'
'000. "ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1/2024. ОТ 01,11:2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.) ПО ДАННЫМ ПОКУЛАТЕЛЯ'


In [214]:
_LAT_TO_CYR = str.maketrans("ABCEHKMOPTXY", "АВСЕНКМОРТХУ")

RE_SMART_LAT = re.compile(r"(?<=[А-ЯЁ])[A-Z]|[A-Z](?=[А-ЯЁ])")

def normalize_text(text: str) -> str:
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.upper().translate(m)
    text = RE_NUMERIC.sub(_fix_numeric, text)        # сначала цифры
    text = RE_SMART_LAT.sub(                          # потом латиница в кирилл. контексте
        lambda m: m.group(0).translate(_LAT_TO_CYR), text
    )
    text = _apply_rules(text, _RULES_SPACE)
    text = _apply_rules(text, _RULES_QUOTES)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t)))

Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ. ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20:10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025R.'
'(PY6.)'
'000. "ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1/2024. ОТ 01,11:2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.) ПО ДАННЫМ ПОКУЛАТЕЛЯ'


In [215]:
# 1. Сначала защищаем числовой контекст: цифра-пробел-ООО-точка/запятая-цифра
RE_OOO_THOUSANDS = re.compile(r'(?<=\d )[О0]{3}(?=[.,]\d)')  # 20 ООО.25 -> 20 000.25

# 2. Потом заменяем ООО как форму собственности
RE_OOO_COMPANY = re.compile(r'[О0]{3}(?=[\s.]*[«"""]|\s+[А-ЯЁ])') # 000 "Рога" -> ООО "Рога"

def normalize_text(text: str) -> str:
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.upper().translate(m)
    text = RE_OOO_THOUSANDS.sub('000', text)  # защищаем числовой контекст
    text = RE_OOO_COMPANY.sub('ООО', text)    # заменяем ООО как форму собственности
    text = RE_NUMERIC.sub(_fix_numeric, text)
    text = RE_SMART_LAT.sub(
        lambda m: m.group(0).translate(_LAT_TO_CYR), text
    )
    text = _apply_rules(text, _RULES_SPACE)
    text = _apply_rules(text, _RULES_QUOTES)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t)))


Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ. ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20:10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025R.'
'(PY6.)'
'ООО. "ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1/2024. ОТ 01,11:2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.) ПО ДАННЫМ ПОКУЛАТЕЛЯ'


In [ ]:
RE_DATE = re.compile(r'(\d{1,2})[^\s\d](\d{1,2})[^\s\d](\d{2}(?:\d{2})?)([А-ЯЁA-Z]\.?)?')

def _fix_date(m: re.Match) -> str:
    result = f"{m.group(1)}.{m.group(2)}.{m.group(3)}"
    if m.group(4):
        result += " Г."
    return result


def normalize_text(text: str) -> str:
    maps = [
        NORM_MAP,
        _build_extended_cyrillic_map(),
    ]
    for m in maps:
        text = text.upper().translate(m)
    text = RE_DATE.sub(_fix_date, text)  # сначала даты
    text = RE_OOO_THOUSANDS.sub('000', text)  # потом числовой контекст
    text = RE_OOO_COMPANY.sub('ООО', text)    # потом ООО как форма собственности
    text = RE_NUMERIC.sub(_fix_numeric, text) # потом остальные числа
    text = RE_SMART_LAT.sub(
        lambda m: m.group(0).translate(_LAT_TO_CYR), text
    )
    text = _apply_rules(text, _RULES_SPACE)
    text = _apply_rules(text, _RULES_QUOTES)
    return text.strip()

print("Нормализованные строки:")
for t in raw_texts:
    print(repr(normalize_text(t)))

Нормализованные строки:
'ПРИЛОЖЕНИЕ 2 К ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ ОБЪЕКТОВ; ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ'
'АКТ СВЕРКИ РАСЧЕТОВ'
'ПО ДОГОВОРУ О ПРЕДОСТАВЛЕНИИ МОЩНОСТИ КВАЛИФИЦИРОВАННЫХ ГЕНЕРИРУЮЩИХ. ОБЪЕКТОВ, ФУНКЦИОНИРУЮЩИХ НА ОСНОВЕ ИСПОЛЬЗОВАНИЯ ВОЗОБНОВЛЯЕМЫХ ИСТОЧНИКОВ ЭНЕРГИИ N DPMV2-S-2H000341-RUSALBAZ-GVIE1862-21 ОТ 20.10.2021 Г. ЗА 1 КВАРТАЛ 2025 Г.'
'00 "ЮНИГРИН ПАУЗР" МЫ, НИЖЕПОДГИСАВШИЕСЯ, С ОДНОЙ СТОРОНЫ, И ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ:'
'01.04.2025 Г.'
'(PY6.)'
'ООО. "ЮНИГРИН ПАУЗР"'
'ОТ ПРОДААЦА'
'ОТ ЛОКУПАТЕЛЯ ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД"'
'ПО ДОВ-ТИ №124.1.2024. ОТ 01.11.2024 САВИНА Т.Г:'
'ВЕДУЩИЙ БУХГАЛТЕР ПО УЧЕТУ ОПЕРАЦИЙ НА ОРЭМ САВИНА Т.Г.:'
'_'
'ПО ДАННЫМ ПРОДАВЦА'
'(PY6.) ПО ДАННЫМ ПОКУЛАТЕЛЯ'